In [ ]:
%env PYTHONHASHSEED=0
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import numpy as np
import scanpy as sc
from anticor_features.anticor_features import get_anti_cor_genes
import rpy2

In [ ]:
np.random.seed(0)
sc.set_figure_params(dpi = 300, dpi_save = 300, frameon = False)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(ggplot2)
library(ggpubr)
library(SingleCellExperiment)
library(eulerr)
library(clustree)
library(mrtree)
library(CHOIR)

In [ ]:
# load anndata object
human = sc.read("../output/human/human_adata_object_filtered.h5ad")

In [ ]:
human

In [ ]:
# plot UMI % for most highly expressed genes
sc.pl.highest_expr_genes(human, n_top=20)

In [ ]:
# save counts to layer
human.layers['raw_counts'] = human.X.copy()

In [ ]:
# library normalization and log1p transformation
scales_counts = sc.pp.normalize_total(human, target_sum=None, inplace=False)
human.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)
human.X = human.layers["log1p_norm"]

In [ ]:
# calculate top 2,000 highly variable genes using seurat method, and save results to CSV file
sc.pp.highly_variable_genes(human, flavor = "seurat", batch_key = "sampleid", n_top_genes = 2000)
human_hvgs = human.var[human.var['highly_variable'] == True].index.values
human.var[human.var['highly_variable'] == True].to_csv("../output/human/human_highly_variable_genes_dispersion.csv")
human.var[human.var['highly_variable'] == True].to_pickle("../output/human/human_highly_variable_genes_dispersion.pkl")

In [ ]:
%%time

np.random.seed(0)

# calculate anti-correlated genes; an alternative method of feature selection
anti_cor_table = get_anti_cor_genes(human.X.T,
                                    human.var.index.tolist(),
                                    species="hsapiens")
anti_cor_table['selected'] = anti_cor_table['selected'].fillna(False)

In [ ]:
# add anti-correlated feature table to human.var metadata
human.var = pd.concat([human.var,anti_cor_table], axis=1)

In [ ]:
dispersion_hvgs = human.var[human.var.highly_variable == True].index.values

In [ ]:
anti_cor_hvgs = anti_cor_table[anti_cor_table.selected == True].index.values

In [ ]:
%%R -i dispersion_hvgs -i anti_cor_hvgs -w 6 -h 4 -r 300 --units in

# evaluate overlap between top 2,000 dispersion-based HVGs and significant anti-correlated features
plot(euler(c("Dispersion" = length(setdiff(dispersion_hvgs, anti_cor_hvgs)), "Anti-correlation" = length(setdiff(anti_cor_hvgs, dispersion_hvgs)), 
     "Dispersion&Anti-correlation" = length(intersect(dispersion_hvgs, anti_cor_hvgs)))),
    fills = c("lightblue", "darkgoldenrod1"),
     edges = TRUE,
     fontsize = 8,
     quantities = list(fontsize = 10))

In [ ]:
# use anti-correlated features as highly variable genes for downstream tasks
human.var.highly_variable = human.var.selected

In [ ]:
# save final HVG metadata to CSV file
human.var[human.var['highly_variable'] == True].to_csv("../output/human/human_anticor_features.csv")
human.var[human.var['highly_variable'] == True].to_pickle("../output/human/human_anticor_features.pkl")

In [ ]:
# scale data
sc.pp.scale(human, max_value=10)
human.layers["scale"] = human.X.copy()

In [ ]:
# run PCA
sc.tl.pca(human, n_comps = 100, svd_solver='arpack', use_highly_variable = True)

In [ ]:
# evaluate variance explained by PCs
sc.pl.pca_variance_ratio(human, log=True, n_pcs = 100)

In [ ]:
# evaluate feature loadings for top PCs
sc.pl.pca_loadings(human, components = '1,2,3')

In [ ]:
# create KNN graph
sc.pp.neighbors(human, n_pcs=90, use_rep = "X_pca", n_neighbors = 15)

In [ ]:
# run UMAP
sc.tl.umap(human)
human.obsm['X_umap_original'] = human.obsm['X_umap'].copy()

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (6, 5), "grid.alpha":0}):
    sc.pl.umap(human, color=['sampleid'], frameon=False, title='',
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"})

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}):
    sc.pl.umap(human, color=['sampleid'], frameon=False, title='',
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
              legend_loc='none')

In [ ]:
# run harmony integration to correct for batch and sample differences
harmony_args = {'theta': [2, 2], 'max_iter_harmony':30}
sc.external.pp.harmony_integrate(human, key = ["batch", 'sampleid'], basis='X_pca', 
                                 adjusted_basis='X_pca_harmony', **harmony_args
                                )

In [ ]:
# create harmony-corrected KNN graph
sc.pp.neighbors(human, n_pcs=90, use_rep = "X_pca_harmony", key_added = "harmony", n_neighbors = 15)

In [ ]:
# run UMAP
sc.tl.umap(human, neighbors_key = "harmony")
human.obsm['X_umap_harmony'] = human.obsm['X_umap'].copy()

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (6, 5), "grid.alpha":0}):
    sc.pl.umap(human, color=['sampleid'], frameon=False, title='',
               size = 5,
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"})

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}):
    sc.pl.umap(human, color=['sampleid'], frameon=False, title='',
               size = 5,
        palette={"Control_A":"#0e427c", "Control_B":"#5a97c1", "Control_C":"#74c8c3",
                 "CART_A":"#6f1806", "CART_B":"#dd2c09", "CART_C":"#e67424", "CART_D":"#f5c34d"},
              legend_loc='none')

In [ ]:
# this function will run leiden clustering across a range of resolution values
def multires_leiden(adata, resolutions, neighbors_key):
    for x in resolutions:
        print("Finding clusters at Leiden resolution " + str(x) + " ...")
        sc.tl.leiden(adata, key_added = "leiden_" + str(x), resolution = x, neighbors_key = neighbors_key)
        print("Found " + str(len(adata.obs["leiden_" + str(x)].cat.categories.values)) + " clusters ...")

In [ ]:
# run the multi-resolution clustering
resolutions = np.arange(0, 4.1, 0.1).round(decimals=1)
multires_leiden(human, resolutions, 'harmony')

In [ ]:
# create cluster dataframe from which tree will be built
leiden_keys = ['leiden_' + str(x) for x in resolutions]
cluster_tree = human.obs[leiden_keys]

In [ ]:
# extract nearest neighbors connectivities matrix and reduction embeddings
nn_mat = human.obsp['harmony_connectivities'].todense()
reduction = human.obsm['X_pca_harmony']
reduction = reduction[:, 0:90]

In [ ]:
# extract normalized count matrix, cell barcodes, and gene names; subsetting for HVGs
input_matrix = human.layers['log1p_norm'].T
input_matrix = input_matrix[human.var.highly_variable, :].todense()
cell_ids = np.asarray(human.obs_names)
gene_ids = np.asarray(human.var_names)
gene_ids = gene_ids[human.var.highly_variable]

In [ ]:
%%R -i cluster_tree
# construct a hierarchical cluster tree using MRtree 
set.seed(0)
hier_cluster_tree = mrtree(cluster_tree, prefix='leiden_', n.cores = 48)

In [ ]:
%%R -i input_matrix -i gene_ids -i cell_ids
# create a SingleCellExperiment object from anndata inputs
rownames(input_matrix) <- gene_ids
colnames(input_matrix) <- cell_ids
sce <- SingleCellExperiment(assays = list(logcounts = input_matrix))

In [ ]:
%%R

sce

In [ ]:
%%R -i nn_mat -i reduction

# add column/row names to matrices
rownames(nn_mat) <- cell_ids
colnames(nn_mat) <- cell_ids
rownames(reduction) <- cell_ids

In [ ]:
%%R
# extract hierarchical cluster tree
hc_tree = hier_cluster_tree$labelmat.mrtree

In [ ]:
%%R

#' Below is a modified version of the pruneTree function from the CHOIR package,
#' which will prune the hierarchical clustering tree using random forest classifiers
#' and permutation testing to identify a final set of clusters. This function has been
#' edited from the original package code to make it compatible with a SingleCellExperiment
#' object derived from a scanpy anndata object's leiden clustering results, after a clustering
#' tree has been generated with mrtree. These adjustments increase the function's speed by 
#' removing the step checking for underclustering, avoiding calls to Seurat's FindClusters 
#' function. Additional functionality/options of the original pruneTree function aside from
#' this use case may no longer work in this implementation.
#' 
#' We name this modified version of the function OverpruneTree (as only overclustering is 
#' evaluated in this implementation.
#' 
#' The documentation below is copied from the CHOIR package, with the changes described above.
#'
#' ----------
#'
#' Prune clustering tree using random forest classifiers
#'
#' To identify a final set of clusters, this function will move iteratively from
#' the bottom up to prune the provided hierarchical clustering tree using a
#' framework of random forest classifiers and permutation tests.
#'
#' If \code{CHOIR:::buildTree()} was run prior to this function, most parameters
#' will be retrieved from the object. Alternately, parameter values can be
#' supplied. For multi-modal data, optionally supply parameter inputs as
#' vectors/lists that sequentially specify the value for each modality.
#'
#' @param object An object of class 'Seurat', 'SingleCellExperiment', or
#' 'ArchRProject'.
#' @param key The name under which CHOIR-related data for this run is stored in
#' the object. Defaults to 'CHOIR'.
#' @param alpha A numeric value indicating the significance level used for
#' permutation test comparisons of cluster prediction accuracies. Defaults to
#' 0.05.
#' @param p_adjust A string indicating which multiple comparison adjustment to
#' use. Permitted values are 'bonferroni', 'fdr', and 'none'. Defaults to
#' 'bonferroni'.
#' @param feature_set A string indicating whether to train random forest
#' classifiers on 'all' features or only variable ('var') features. Defaults to
#' 'var'.
#' @param exclude_features A character vector indicating features that should be
#' excluded from input to the random forest classifier. Default = \code{NULL}
#' will not exclude any features.
#' @param n_iterations A numeric value indicating the number of iterations run
#' for each permutation test comparison. Defaults to 100.
#' @param n_trees A numeric value indicating the number of trees in each random
#' forest. Defaults to 50.
#' @param use_variance A boolean value indicating whether to use the variance of
#' the random forest accuracy scores as part of the permutation test threshold.
#' Defaults to \code{TRUE}.
#' @param min_accuracy A numeric value indicating the minimum accuracy required
#' of the random forest classifier, below which clusters will be automatically
#' merged. Defaults to 0.5 (chance).
#' @param min_connections A numeric value indicating the minimum number of
#' nearest neighbors between two clusters for them to be considered 'adjacent'.
#' Non-adjacent clusters will not be merged. Defaults to 1.
#' @param max_repeat_errors Used to account for situations in which random
#' forest classifier errors are concentrated among a few cells that are
#' repeatedly misassigned. A numeric value indicating the maximum number of such
#' 'repeat errors' that will be taken into account. If set to 0, 'repeat errors'
#' will not be evaluated. Defaults to 20.
#' @param distance_approx A boolean value indicating whether or not to use
#' approximate distance calculations. Default = \code{TRUE} will use
#' centroid-based distances.
#' @param distance_awareness A numeric value representing the distance threshold
#' above which a cluster will not merge with another cluster. Specifically,
#' this value is multiplied by the distance between a cluster and its
#' closest distinguishable neighbor to set the threshold. Default = 2 sets
#' this threshold at a 2-fold increase in distance. Alternately, to omit all
#' distance calculations, set to \code{FALSE}.
#' @param collect_all_metrics A boolean value indicating whether to collect and
#' save additional metrics from the random forest classifier comparisons,
#' including feature importances and tree depth. Defaults to \code{FALSE}.
#' @param sample_max A numeric value indicating the maximum number of cells used
#' per cluster to train/test each random forest classifier. Default = \code{Inf}
#' does not cap the number of cells used.
#' @param downsampling_rate A numeric value indicating the proportion of cells
#' used per cluster to train/test each random forest classifier. Default =
#' "auto" sets the downsampling rate according to the dataset size, for
#' efficiency.
#' @param normalization_method A character string or vector indicating which
#' normalization method to use. In general, input data should be supplied to
#' CHOIR after normalization, except in cases when the user wishes to use
#' \code{Seurat::SCTransform()} normalization. Permitted values are 'none' or
#' 'SCTransform'. Defaults to 'none'.
#' @param batch_correction_method A character string or vector indicating which
#' batch correction method to use. Permitted values are 'Harmony' and
#' 'none'. Defaults to 'none'.
#' @param batch_labels If applying batch correction, a character string or
#' vector indicating the name of the column containing the batch labels.
#' Defaults to \code{NULL}.
#' @param cluster_params A list of additional parameters to be passed to
#' Seurat::FindClusters() for clustering at each level of the tree. Note that if
#' \code{group.singletons} is set to \code{TRUE}, \code{CHOIR} relabels initial
#' clusters such that each singleton constitutes its own cluster.
#' @param use_assay For Seurat or SingleCellExperiment objects, a character
#' string or vector indicating the assay(s) to use in the provided object.
#' Default = \code{NULL} will choose the current active assay for Seurat objects
#' and the \code{logcounts} assay for SingleCellExperiment objects.
#' @param cluster_tree An optional dataframe containing the cluster IDs of each
#' cell across the levels of a hierarchical clustering tree. Default = \code{NULL}
#' will use the hierarchical clustering tree generation by function
#' \code{buildTree()}.
#' @param input_matrix An optional matrix containing the feature x cell data on
#' which to train the random forest classifiers. Default = \code{NULL} will use
#' the feature x cell matri(ces) indicated by function \code{buildTree()}.
#' @param nn_matrix An optional matrix containing the nearest neighbor adjacency
#' of the cells. Default = \code{NULL} will look for the adjacency matri(ces)
#' generated by function \code{buildTree()}.
#' @param dist_matrix An optional distance matrix of cell to cell distances (based
#' on dimensionality reduction cell embeddings). Default = \code{NULL} will look
#' for the distance matri(ces) generated by function \code{buildTree()}.
#' @param reduction An optional matrix of dimensionality reduction cell
#' embeddings to be used for distance calculations. Defaults = \code{NULL} will
#' look for the dimensionality reductions generated by function \code{buildTree()}.
#' @param n_cores A numeric value indicating the number of cores to use for
#' parallelization. Default = \code{NULL} will use the number of available cores
#' minus 2.
#' @param random_seed A numeric value indicating the random seed to be used.
#' @param verbose A boolean value indicating whether to use verbose output
#' during the execution of this function. Can be set to \code{FALSE} for a
#' cleaner output.
#'
#' @return Returns the object with the following added data stored under the
#' provided key: \describe{
#'   \item{clusters}{Final clusters and stepwise cluster results for each
#'   progressive pruning step}
#'   \item{parameters}{Record of parameter values used}
#'   \item{records}{Metadata for all recorded permutation test comparisons and
#'   feature importance scores from all comparisons}
#'   }
#'
#' @export
#'
OverpruneTree <- function(object,
                      key = "CHOIR",
                      alpha = NULL,
                      p_adjust = NULL,
                      feature_set = NULL,
                      exclude_features = NULL,
                      n_iterations = NULL,
                      n_trees = NULL,
                      use_variance = NULL,
                      min_accuracy = NULL,
                      min_connections = NULL,
                      max_repeat_errors = NULL,
                      distance_approx = NULL,
                      distance_awareness = 2,
                      collect_all_metrics = FALSE,
                      sample_max = NULL,
                      downsampling_rate = NULL,
                      normalization_method = NULL,
                      batch_correction_method = NULL,
                      batch_labels = NULL,
                      cluster_params = NULL,
                      use_assay = NULL,
                      cluster_tree = NULL,
                      input_matrix = NULL,
                      nn_matrix = NULL,
                      dist_matrix = NULL,
                      reduction = NULL,
                      n_cores = NULL,
                      random_seed = NULL,
                      verbose = TRUE) {

  # ---------------------------------------------------------------------------
  # Retrieve/check parameter input validity
  # ---------------------------------------------------------------------------

  CHOIR:::.validInput(object, "object", "pruneTree")
  CHOIR:::.validInput(key, "key", list("pruneTree", object))
  CHOIR:::.validInput(distance_awareness, "distance_awareness")
  CHOIR:::.validInput(collect_all_metrics, "collect_all_metrics")
  CHOIR:::.validInput(n_cores, "n_cores")
  CHOIR:::.validInput(verbose, "verbose")

  # Set defaults
  if (is.null(n_cores)) {
    n_cores <- parallel::detectCores() - 2
  }
  # Random seed reproducibility
  if (n_cores > 1) {
    RNGkind("L'Ecuyer-CMRG")
  }

  # Retrieve parameter values from buildTree
  buildTree_parameters <- CHOIR:::.retrieveData(object,
                                        key,
                                        "parameters",
                                        "buildTree_parameters")
  default_parameters <- list("alpha" = 0.05,
                             "p_adjust" = "bonferroni",
                             "feature_set" = "var",
                             "exclude_features" = NULL,
                             "n_iterations" = 100,
                             "n_trees" = 50,
                             "use_variance" = TRUE,
                             "min_accuracy" = 0.5,
                             "min_connections" = 1,
                             "max_repeat_errors" = 20,
                             "distance_approx" = TRUE,
                             "sample_max" = Inf,
                             "downsampling_rate" = "auto",
                             "normalization_method" = "none",
                             "batch_correction_method" = "none",
                             "batch_labels" = NULL,
                             "cluster_params" = list(algorithm = 1,
                                                     group.singletons = TRUE),
                             "use_assay"  = NULL,
                             "random_seed" = 1)

  # For any parameters set to NULL, use parameters from buildTree or defaults
  alpha <- CHOIR:::.retrieveParam(alpha, "alpha", buildTree_parameters, default_parameters)
  p_adjust <- CHOIR:::.retrieveParam(p_adjust, "p_adjust", buildTree_parameters, default_parameters)
  feature_set <- CHOIR:::.retrieveParam(feature_set, "feature_set", buildTree_parameters, default_parameters)
  exclude_features <- CHOIR:::.retrieveParam(exclude_features, "exclude_features", buildTree_parameters, default_parameters)
  n_iterations <- CHOIR:::.retrieveParam(n_iterations, "n_iterations", buildTree_parameters, default_parameters)
  n_trees <- CHOIR:::.retrieveParam(n_trees, "n_trees", buildTree_parameters, default_parameters)
  use_variance <- CHOIR:::.retrieveParam(use_variance, "use_variance", buildTree_parameters, default_parameters)
  min_accuracy <- CHOIR:::.retrieveParam(min_accuracy, "min_accuracy", buildTree_parameters, default_parameters)
  min_connections <- CHOIR:::.retrieveParam(min_connections, "min_connections", buildTree_parameters, default_parameters)
  max_repeat_errors <- CHOIR:::.retrieveParam(max_repeat_errors, "max_repeat_errors", buildTree_parameters, default_parameters)
  distance_approx <- CHOIR:::.retrieveParam(distance_approx, "distance_approx", buildTree_parameters, default_parameters)
  sample_max <- CHOIR:::.retrieveParam(sample_max, "sample_max", buildTree_parameters, default_parameters)
  downsampling_rate <- CHOIR:::.retrieveParam(downsampling_rate, "downsampling_rate", buildTree_parameters, default_parameters)
  normalization_method <- CHOIR:::.retrieveParam(normalization_method, "normalization_method", buildTree_parameters, default_parameters)
  batch_correction_method <- CHOIR:::.retrieveParam(batch_correction_method, "batch_correction_method", buildTree_parameters, default_parameters)
  batch_labels <- CHOIR:::.retrieveParam(batch_labels, "batch_labels", buildTree_parameters, default_parameters)
  cluster_params <- CHOIR:::.retrieveParam(cluster_params, "cluster_params", buildTree_parameters, default_parameters)
  use_assay <- CHOIR:::.retrieveParam(use_assay, "use_assay", buildTree_parameters, default_parameters)
  random_seed <- CHOIR:::.retrieveParam(random_seed, "random_seed", buildTree_parameters, default_parameters)
  # Verify parameter validity
  CHOIR:::.validInput(alpha, "alpha")
  CHOIR:::.validInput(p_adjust, "p_adjust")
  CHOIR:::.validInput(feature_set, "feature_set", list("pruneTree", normalization_method))
  CHOIR:::.validInput(exclude_features, "exclude_features")
  CHOIR:::.validInput(n_iterations, "n_iterations")
  CHOIR:::.validInput(n_trees, "n_trees")
  CHOIR:::.validInput(use_variance, "use_variance")
  CHOIR:::.validInput(min_accuracy, "min_accuracy")
  CHOIR:::.validInput(min_connections, "min_connections")
  CHOIR:::.validInput(max_repeat_errors, "max_repeat_errors")
  CHOIR:::.validInput(sample_max, "sample_max")
  CHOIR:::.validInput(downsampling_rate, "downsampling_rate")
  CHOIR:::.validInput(cluster_params, "cluster_params")
  CHOIR:::.validInput(use_assay, "use_assay", object)
  CHOIR:::.validInput(random_seed, "random_seed")

  # Extract cell IDs
  cell_IDs <- CHOIR:::.getCellIDs(object, use_assay)

  CHOIR:::.validInput(cluster_tree, "cluster_tree", object)
  CHOIR:::.validInput(input_matrix, "input_matrix", cell_IDs)
  CHOIR:::.validInput(nn_matrix, "nn_matrix", cell_IDs)
  CHOIR:::.validInput(dist_matrix, "dist_matrix", object)
  CHOIR:::.validInput(reduction, "reduction", list("pruneTree", object))

  # User supplied data should not be mixed with data from buildTree()
  if (methods::is(distance_awareness, "numeric") & distance_approx == FALSE) {
    if (any(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix), !is.null(dist_matrix))) &
        !all(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix), !is.null(dist_matrix)))) {
      warning("If user-supplied data is provided for any of 'cluster_tree', 'input_matrix', 'nn_matrix', or 'dist_matrix', it should be provided for all four.")
    }
  } else if (methods::is(distance_awareness, "numeric") & distance_approx == TRUE) {
    if (any(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix), !is.null(reduction))) &
        !all(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix), !is.null(reduction)))) {
      warning("If user-supplied data is provided for any of 'cluster_tree', 'input_matrix', 'nn_matrix', or 'reduction', it should be provided for all four.")
    }
  } else {
    if (any(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix))) &
        !all(c(!is.null(cluster_tree), !is.null(input_matrix), !is.null(nn_matrix)))) {
      warning("If user-supplied data is provided for any of 'cluster_tree', 'input_matrix', or 'nn_matrix', it should be provided for all three.")
    }
  }

  # Retrieve subtree data
  if (!is.null(buildTree_parameters)) {
    # Subtree names
    subtree_names <- buildTree_parameters[["subtree_names"]]
    subtree_sizes <- buildTree_parameters[["subtree_sizes"]]
    subtree_names_filtered <- subtree_names[subtree_sizes > 3]
    n_subtrees <- length(subtree_names)
    n_subtrees_filtered <- length(subtree_names_filtered)
  } else {
    # If there are no records of parameters used in buildTree function
    warning("No record of parameters used in buildTree() function.")
    # Subtree names
    subtree_names <- "P0"
    subtree_sizes <- length(cell_IDs)
    subtree_names_filtered <- subtree_names[subtree_sizes > 3]
    n_subtrees <- length(subtree_names)
    n_subtrees_filtered <- length(subtree_names_filtered)
  }

  # ---------------------------------------------------------------------------
  # Prepare object
  # ---------------------------------------------------------------------------
  if (verbose) message(format(Sys.time(), "%Y-%m-%d %X"), " : (Step 1/2) Preparing object..")

  # Extract cluster tree
  if (is.null(cluster_tree)) {
    cluster_tree <- CHOIR:::.retrieveData(object, key, "clusters", "full_tree")
    CHOIR:::.validInput(cluster_tree, "cluster_tree", object)
    cluster_tree_provided <- FALSE
  } else {
    cluster_tree_provided <- TRUE
  }
  # Check whether provided clustering tree is ordered correctly & strictly hierarchical
  cluster_tree <- CHOIR:::.checkHierarchy(cluster_tree)
  # Check whether provided clustering tree has appropriately named clusters
  # If not, rename them
  cluster_tree <- CHOIR:::.checkClusterLabels(cluster_tree)
  rownames(cluster_tree) <- cell_IDs
  # Number of starting clusters
  n_starting_clusters <- dplyr::n_distinct(cluster_tree[, ncol(cluster_tree)])
  # Number of levels in cluster tree
  n_levels <- ncol(cluster_tree)
  # Will change based on multiple comparison correction, if applicable
  adjusted_alpha <- alpha

  # Extract input matrix/matrices
  if (!is.null(input_matrix)) {
    input_matrix_provided <- TRUE
    input_matrix <- CHOIR:::.getMatrix(use_matrix = input_matrix,
                               exclude_features = exclude_features,
                               verbose = FALSE)
    input_matrix <- BiocGenerics::t(input_matrix)
    input_matrices <- list("P0" = input_matrix)
    n_input_matrices <- 1
    # Feature names
    features <- colnames(input_matrix)
    # Clean up
    rm(input_matrix)
  } else if (!is.null(buildTree_parameters)) {
    input_matrix_provided <- FALSE
    # Parameters for extracting matrices
    use_assay <- buildTree_parameters[["use_assay"]]
    use_slot <- buildTree_parameters[["use_slot"]]
    ArchR_matrix <- buildTree_parameters[["ArchR_matrix"]]
    # Number of modalities & object type
    if (methods::is(object, "ArchRProject")) {
      n_modalities <- max(length(ArchR_matrix), 1)
      object_type <- "ArchRProject"
      CHOIR:::.requirePackage("ArchR", installInfo = "Instructions at archrproject.com")
    } else {
      n_modalities <- max(length(use_assay), 1)
      if (methods::is(object, "Seurat")) {
        object_type <- "Seurat"
      } else {
        object_type <- "SingleCellExperiment"
        CHOIR:::.requirePackage("SingleCellExperiment", source = "bioc")
      }
    }
    # Need n_modalities to validate some inputs
    CHOIR:::.validInput(normalization_method, "normalization_method", list(object, n_modalities, use_assay))
    CHOIR:::.validInput(batch_correction_method, "batch_correction_method", list(n_modalities, "pruneTree"))
    CHOIR:::.validInput(batch_labels, "batch_labels", object)
    CHOIR:::.validInput(distance_approx, "distance_approx", list(length(cell_IDs), object_type, n_modalities))

    # If reduction was not recalculated for buildTree, there will only be one input matrix
    if (buildTree_parameters[["subtree_reductions"]] == FALSE) {
      n_input_matrices <- 1
    } else {
      n_input_matrices <- n_subtrees_filtered
    }
    # Initialize list of feature names
    features <- c()
    # Initialize list of input matrices
    input_matrices <- vector(mode = "list", n_subtrees_filtered)

    # Extract data
    for (subtree in 1:n_input_matrices) {
      subtree_name <- subtree_names_filtered[subtree]
      use_cells_subtree <- CHOIR:::.retrieveData(object, key, "cell_IDs", paste0(subtree_name, "_cell_IDs"))
      # Use all / var features
      if (feature_set == "all") {
        use_features_subtree <- NULL
      } else {
        use_features_subtree <- CHOIR:::.retrieveData(object, key, "var_features", paste0(subtree_name, "_var_features"))
      }
      # Extract input matrix for random forest comparisons
      if (n_modalities > 1) {
        input_matrix_list <- vector("list", n_modalities)
        for (m in 1:n_modalities) {
          # Match input arguments
          use_assay_m <- CHOIR:::.matchArg(use_assay, m)
          use_slot_m <- CHOIR:::.matchArg(use_slot, m)
          ArchR_matrix_m <- CHOIR:::.matchArg(ArchR_matrix, m)
          normalization_method_m <- CHOIR:::.matchArg(normalization_method, m)
          # Features
          if (length(use_features_subtree) > 1) {
            use_features_subtree_m <- use_features_subtree[[m]]
          } else {
            use_features_subtree_m <- use_features_subtree
          }
          input_matrix_list[[m]] <- CHOIR:::.getMatrix(object = object,
                                               use_assay = use_assay_m,
                                               use_slot = use_slot_m,
                                               ArchR_matrix = ArchR_matrix_m,
                                               use_features = use_features_subtree_m,
                                               exclude_features = exclude_features,
                                               use_cells = use_cells_subtree,
                                               verbose = FALSE)
          if (normalization_method_m == "SCTransform") {
            input_matrix_list[[m]] <- suppressWarnings(Seurat::SCTransform(Seurat::CreateSeuratObject(input_matrix_list[[m]]),
                                                                           return.only.var.genes = FALSE,
                                                                           seed.use = random_seed,
                                                                           verbose = FALSE)@assays$SCT@scale.data)
          }
        }
        input_matrix <- do.call(rbind, input_matrix_list)
        input_matrix <- BiocGenerics::t(input_matrix)
        # Clean up
        rm(use_features_subtree_m)
        rm(use_cells_subtree_m)
      } else {
        input_matrix <- CHOIR:::.getMatrix(object = object,
                                   use_assay = use_assay,
                                   use_slot = use_slot,
                                   ArchR_matrix = ArchR_matrix,
                                   use_features = use_features_subtree,
                                   exclude_features = exclude_features,
                                   use_cells = use_cells_subtree,
                                   verbose = FALSE)
        if (normalization_method == "SCTransform") {
          input_matrix <- suppressWarnings(Seurat::SCTransform(Seurat::CreateSeuratObject(input_matrix),
                                                               return.only.var.genes = FALSE,
                                                               seed.use = random_seed,
                                                               verbose = FALSE)@assays$SCT@scale.data)
        }
        input_matrix <- BiocGenerics::t(input_matrix)
      }
      # Add to list of input_matrices
      input_matrices[[subtree]] <- input_matrix
      # Add features to vector
      features <- unique(c(features, colnames(input_matrix)))

    }
    names(input_matrices) <- subtree_names_filtered
    # Clean up
    rm(use_features_subtree)
    rm(use_cells_subtree)
    rm(input_matrix)
  } else {
    stop("No 'input_matrix' supplied. Please supply valid input!")
  }

  # Extract nearest neighbor matrix/matrices
  if (!is.null(nn_matrix)) {
    nn_matrix_provided <- TRUE
    nn_matrices <- list("P0" = nn_matrix)
    # Clean up
    rm(nn_matrix)
  } else if (!is.null(buildTree_parameters)) {
    nn_matrix_provided <- FALSE
    # Initialize list of nearest neighbor adjacency matrices
    nn_matrices <- vector(mode = "list", n_subtrees_filtered)
    # For each subtree
    for (subtree in 1:n_subtrees_filtered) {
      subtree_name <- subtree_names_filtered[subtree]
      nn_matrices[[subtree]] <- CHOIR:::.retrieveData(object, key, "graph", paste0(subtree_name, "_graph_nn"))
    }
    names(nn_matrices) <- subtree_names_filtered
  } else {
    stop("No nearest neighbor adjacency matrix provided.")
  }

  # Reduction
  if (!is.null(reduction)) {
    reduction_provided <- TRUE
  } else {
    reduction_provided <- FALSE
  }

  # Distance matrix
  if (!is.null(dist_matrix)) {
    dist_matrix_provided <- TRUE
  } else {
    dist_matrix_provided <- FALSE
  }

  # If 'Harmony' batch correction was used, extract batch labels
  if (batch_correction_method == "Harmony") {
    batches <- CHOIR:::.retrieveData(object = object, key = key, type = "cell_metadata", name = batch_labels)
    if ("Rle" %in% methods::is(batches)) {
      batches <- methods::as(batches, "character")
    }
    names(batches) <- cell_IDs
  }

  # Set downsampling rate
  if (downsampling_rate == "auto") {
    downsampling_rate <- min(1, (1/2)^(log10(length(cell_IDs)/5000)))
  }

  # Report object & parameter details
  if (verbose) message("\nInput data:",
                       "\n - Object type: ", object_type,
                       "\n - # of cells: ", length(cell_IDs),
                       "\n - # of modalities: ", n_modalities,
                       "\n - # of subtrees: ", n_subtrees,
                       "\n - # of levels: ", n_levels,
                       "\n - # of starting clusters: ", n_starting_clusters)
  if (verbose) message("\nProceeding with the following parameters:",
                       "\n - Intermediate data stored under key: ", key,
                       "\n - Alpha: ", alpha,
                       "\n - Multiple comparison adjustment: ", p_adjust,
                       "\n - Features to train RF: ", feature_set,
                       "\n - # of excluded features: ", length(exclude_features),
                       "\n - # of permutations: ", n_iterations,
                       "\n - # of RF trees: ", n_trees,
                       "\n - Use variance: ", use_variance,
                       "\n - Minimum accuracy: ", min_accuracy,
                       "\n - Minimum connections: ", min_connections,
                       "\n - Maximum repeated errors: ", max_repeat_errors,
                       "\n - Distance awareness: ", distance_awareness,
                       "\n - Distance approximation: ", distance_approx,
                       "\n - Maximum cells sampled: ", sample_max,
                       "\n - Downsampling rate: ", round(downsampling_rate, 4),
                       "\n - # of cores: ", n_cores,
                       "\n - Random seed: ", random_seed,
                       "\n")

  # ---------------------------------------------------------------------------
  # Initialize output data structures
  # ---------------------------------------------------------------------------

  # Data frame to track comparisons to avoid redundancy
  all_metrics <- c('comparison', 'cluster1_size', 'cluster2_size', 'sample_size',
                   'mean_accuracy', 'var_accuracy', 'mean_errors',
                   'mean_permuted_accuracy', 'var_permuted_accuracy',
                   'percentile_accuracy', 'percentile_variance',
                   'n_repeat_errors1', 'n_repeat_errors2',
                   'mean_repeat_errors1', 'mean_repeat_errors2',
                   'mean_modified_accuracy', 'var_modified_accuracy',
                   'percentile_modified_accuracy', 'percentile_modified_variance',
                   'batches_used', 'batch_mean_accuracies',
                   'connectivity', 'root_distance', 'subtree_distance', 'time',
                   'decision')
  selected_metrics <- all_metrics[c(1:11,
                                    `if`(collect_all_metrics == TRUE | max_repeat_errors > 0, 12:15, NULL),
                                    `if`(max_repeat_errors > 0, 16:19, NULL),
                                    `if`(batch_correction_method == "Harmony", 20:21, NULL),
                                    `if`(collect_all_metrics == TRUE | min_connections > 0, 22, NULL),
                                    `if`(methods::is(distance_awareness, "numeric"), 23:24, NULL),
                                    25:26)]
  comparison_records <- data.frame(matrix(ncol = length(selected_metrics), nrow = 0))
  colnames(comparison_records) <- selected_metrics

  # Feature importance records
  feature_importance_records <- data.frame(matrix(ncol = (length(features)+2), nrow = 0))
  colnames(feature_importance_records) <- c('cluster1', 'cluster2', features)

  # Record of distances
  if (methods::is(distance_awareness, "numeric")) {
    distance_records <- data.frame(cluster_name = NULL,
                                   min_root_distance = NULL,
                                   min_subtree_distance = NULL,
                                   max_pval = NULL)
  } else {
    distance_records <- NULL
  }

  # Record of stepwise cluster IDs
  stepwise_cluster_IDs <- data.frame(CellID = cell_IDs)

  # Record underclustering checks
  checked_for_underclustering <- c()
  results_of_underclustering_check <- c()
  underclustering_buffer <- FALSE

  # -------------------------------------------------------------------------
  # Iterate through each level of the cluster tree (bottom-up)
  # -------------------------------------------------------------------------

  # Track progress
  if (verbose) message(format(Sys.time(), "%Y-%m-%d %X"), " : (Step 2/2) Iterating through clustering tree..")

  # Get mean cluster size at each level of the tree
  mean_cluster_sizes <- apply(cluster_tree, 2, FUN = function(x) length(cell_IDs)/dplyr::n_distinct(x))
  level_weights <- rep(1, n_levels)
  level_weights[mean_cluster_sizes > 0.1*length(cell_IDs)] <- 5
  level_weights <- (level_weights/sum(level_weights))*95
  names(level_weights) <- paste0("L", seq(0, n_levels - 1))

  # Progress bar
  start_time <- Sys.time()
  hour_start_time <- Sys.time()
  pb <- progress::progress_bar$new(format = "Pruning tree..        [:bar] (:percent) in :elapsedfull",
                                   total = 100, clear = FALSE)
  pb$tick(0)
  percent_done <- 0

  # Starting values
  parent_IDs <- cluster_tree[, n_levels-1]
  child_IDs <- cluster_tree[, n_levels]
  n_current_clusters <- dplyr::n_distinct(child_IDs)
  # Start at bottom 2 levels of tree
  lvl <- n_levels-1
  # Complete?
  complete <- FALSE
  # Progress markers
  progress_markers <- c(10,20,30,40,50,60,70,80,90)

  while (complete == FALSE) {
    # Get all cluster IDs at this level
    unique_parent_IDs <- unique(parent_IDs)
    evolving_parent_IDs <- unique_parent_IDs

    # For each parent cluster
    for (parent in 1:length(unique_parent_IDs)) {
      # Current parent cluster
      parent_cluster <- unique_parent_IDs[parent]
      parent_inds <- which(parent_IDs == parent_cluster)
      # All child clusters of this parent
      unique_child_IDs <- unique(child_IDs[parent_inds])
      # If there is only one child, move on (this cluster branch does not split at this level of the tree)
      # If there are two or more child clusters, start to make comparisons
      n_child_clusters <- length(unique_child_IDs)

      if (n_child_clusters > 1) {
        # Create matrix for comparison results
        result_matrix <- matrix(NA, n_child_clusters, n_child_clusters)
        colnames(result_matrix) <- unique_child_IDs
        rownames(result_matrix) <- unique_child_IDs
        # For each child cluster
        for (child1 in 1:(n_child_clusters-1)) {
          # Name of child cluster 1
          child1_name <- unique_child_IDs[child1]
          # Get names of cells belonging to child1 cluster
          child1_cells <- cell_IDs[which(child_IDs == child1_name)]
          # If there is only one cell in child1, simply fill in the result_matrix with all "merge"
          if (length(child1_cells) == 1) {
            result_matrix[child1_name, ] <- "merge"
            result_matrix[, child1_name] <- "merge"
            result_matrix[child1_name, child1_name] <- NA
          } else {
            # Compare the child cluster pairwise to each subsequent sibling cluster
            for (child2 in (child1+1):n_child_clusters) {
              comparison_start_time <- Sys.time()
              child2_name <- unique_child_IDs[child2]
              # Get names of cells belonging to child1 cluster
              child2_cells <- cell_IDs[which(child_IDs == child2_name)]
              # If there is only 1 cell in child2, automatically merge
              if (length(child2_cells) == 1) {
                result_matrix[child1_name, child2_name] <- "merge"
                result_matrix[child2_name, child1_name] <- "merge"
              } else {
                # Check whether clusters have been previously compared
                previous_comparison <- CHOIR:::.checkComparisonRecords(cluster1_name = child1_name,
                                                               cluster1_cells = child1_cells,
                                                               cluster2_name = child2_name,
                                                               cluster2_cells = child2_cells,
                                                               comparison_records = comparison_records)
                # If they have, fill in result matrix
                if (previous_comparison[["previously_compared"]] == TRUE) {
                  # Add result to matrix
                  result_matrix[child1_name, child2_name] <- previous_comparison[["result"]]
                  result_matrix[child2_name, child1_name] <- previous_comparison[["result"]]
                } else {
                  # Determine which input & nn matrices to use
                  roots <- unlist(stringr::str_extract_all(c(child1_name, child2_name), "P\\d*"))
                  if (dplyr::n_distinct(roots) == 1) {
                    if (n_input_matrices > 1) {
                      use_input_matrix <- roots[1]
                    } else {
                      use_input_matrix <- "P0"
                    }
                    use_nn_matrix <- roots[1]
                  } else {
                    use_input_matrix <- "P0"
                    use_nn_matrix <- "P0"
                  }
                  # Check whether distance between current clusters is greater than previous comparisons
                  distance_check <- CHOIR:::.checkDistance(object = object,
                                                   key = key,
                                                   cluster1_name = child1_name,
                                                   cluster1_cells = child1_cells,
                                                   cluster2_name = child2_name,
                                                   cluster2_cells = child2_cells,
                                                   distance_awareness = distance_awareness,
                                                   distance_approx = distance_approx,
                                                   use_input_matrix = use_input_matrix,
                                                   dist_matrix = dist_matrix,
                                                   reduction = reduction,
                                                   distance_records = distance_records)
                  if (distance_check[["distance_conflict"]] == TRUE) {
                    # Add result to matrix
                    result_matrix[child1_name, child2_name] <- "split"
                    result_matrix[child2_name, child1_name] <- "split"
                    # Add result to comparison records
                    current_comparison <- matrix(rep(NA, ncol(comparison_records)), nrow = 1, ncol = ncol(comparison_records))
                    colnames(current_comparison) <- colnames(comparison_records)
                    current_comparison <- data.frame(current_comparison)
                    current_comparison$comparison <- paste0(child1_name, " vs. ", child2_name)
                    current_comparison$cluster1_size <- length(child1_cells)
                    current_comparison$cluster2_size <- length(child2_cells)
                    current_comparison$root_distance <- distance_check[["P0_distance"]]
                    current_comparison$subtree_distance <- distance_check[["P_i_distance"]]
                    current_comparison$decision <- "split: distance"
                    current_comparison$time <- round(difftime(Sys.time(), comparison_start_time, units = "secs"), 2)
                    comparison_records <- rbind(comparison_records, current_comparison)
                  } else if (distance_check[["distance_conflict"]] == FALSE) {
                    # Run comparison
                    comparison_output <- CHOIR:::.runPermutationTest(cluster1_name = child1_name,
                                                             cluster1_cells = child1_cells,
                                                             cluster1_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                          batches[child1_cells],
                                                                                          NULL),
                                                             cluster2_name = child2_name,
                                                             cluster2_cells = child2_cells,
                                                             cluster2_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                          batches[child2_cells],
                                                                                          NULL),
                                                             alpha = ifelse(p_adjust != "none", adjusted_alpha, alpha),
                                                             n_iterations = n_iterations,
                                                             n_trees = n_trees,
                                                             use_variance = use_variance,
                                                             min_accuracy = min_accuracy,
                                                             min_connections = min_connections,
                                                             max_repeat_errors = max_repeat_errors,
                                                             collect_all_metrics = collect_all_metrics,
                                                             sample_max = sample_max,
                                                             downsampling_rate = downsampling_rate,
                                                             input_matrix = input_matrices[[use_input_matrix]],
                                                             nn_matrix = nn_matrices[[use_nn_matrix]],
                                                             comparison_records = comparison_records,
                                                             feature_importance_records = feature_importance_records,
                                                             P0_distance = distance_check[["P0_distance"]],
                                                             P_i_distance = distance_check[["P_i_distance"]],
                                                             comparison_start_time = comparison_start_time,
                                                             n_cores = n_cores,
                                                             random_seed = random_seed)
                    # Add result to matrix
                    result_matrix[child1_name, child2_name] <- comparison_output[["result"]]
                    result_matrix[child2_name, child1_name] <- comparison_output[["result"]]
                    # If split, add distances to distance_records
                    if (comparison_output[["result"]] == "split" & methods::is(distance_awareness, "numeric")) {
                      distance_records <- CHOIR:::.addDistance(cluster1_name = child1_name,
                                                       cluster2_name = child2_name,
                                                       P0_distance = distance_check[["P0_distance"]],
                                                       P_i_distance = distance_check[["P_i_distance"]],
                                                       max_p = comparison_output[["max_p"]],
                                                       distance_records = distance_records)
                    }
                    # Update records
                    comparison_records <- comparison_output[["comparison_records"]]
                    feature_importance_records <- comparison_output[["feature_importance_records"]]
                  }
                }
              }
              if (lvl >= 0) {
                tick_amount <- (1/(n_child_clusters - child1))*0.9*(1/(n_child_clusters-1))*0.9*(1/length(unique_parent_IDs))*(0.9*level_weights[paste0("L", lvl)])
                pb$tick(tick_amount)
                if (verbose & ((((percent_done + tick_amount) %/% 10) - (percent_done %/% 10) > 0) |
                               (difftime(Sys.time(), hour_start_time, units = "hours") >= 1))) {
                  hour_start_time <- Sys.time()
                  pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"),
                                    " : ", round((percent_done + tick_amount)), "% (", n_levels - lvl, "/", n_levels ," levels) in ",
                                    round(difftime(Sys.time(), start_time, units = "min"), 2),
                                    " min. ", dplyr::n_distinct(child_IDs), " clusters remaining."))
                }
                percent_done <- percent_done + tick_amount
              }
            }
          }
          if (lvl >= 0) {
            tick_amount <- 0.1*(1/(n_child_clusters-1))*0.9*(1/length(unique_parent_IDs))*(0.9*level_weights[paste0("L", lvl)])
            pb$tick(tick_amount)
            if (verbose & ((((percent_done + tick_amount) %/% 10) - (percent_done %/% 10) > 0) |
                           (difftime(Sys.time(), hour_start_time, units = "hours") >= 1))) {
              hour_start_time <- Sys.time()
              pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"),
                                " : ", round((percent_done + tick_amount)), "% (", n_levels - lvl, "/", n_levels ," levels) in ",
                                round(difftime(Sys.time(), start_time, units = "min"), 2),
                                " min. ", dplyr::n_distinct(child_IDs), " clusters remaining."))
            }
            percent_done <- percent_done + tick_amount
          }
        }
        # Identify which clusters will merge & update child IDs
        # If all clusters will merge
        if (sum(result_matrix == "merge", na.rm = TRUE) == n_child_clusters*(n_child_clusters - 1)) {
          # Update child IDs
          child_IDs[parent_inds] <- parent_IDs[parent_inds]
        } else {
          # For each child
          for (child in 1:n_child_clusters) {
            child_name <- unique_child_IDs[child]
            # Count comparisons
            n_merge <- sum(result_matrix[child_name,] == "merge", na.rm = TRUE)
            # If there is more than one cluster that is slated to merge
            if (n_merge > 1) {
              # Identify the set of clusters this cluster is slated to merge with
              partner_clusters <- colnames(result_matrix[, colnames(result_matrix) != child_name])[result_matrix[child_name, colnames(result_matrix) != child_name] == "merge"]
              # Check whether these partner clusters are also slated to merge with each other
              for (partner1 in 1:(length(partner_clusters)-1)) {
                partner1_name <- partner_clusters[partner1]
                for (partner2 in (partner1 + 1):length(partner_clusters)) {
                  partner2_name <- partner_clusters[partner2]
                  # If not, assess whether to bridge the clusters or not
                  if (result_matrix[partner1_name, partner2_name] == "split") {
                    # Get cell IDs belonging to each cluster
                    child_cells <- cell_IDs[which(child_IDs == child_name)]
                    partner1_cells <- cell_IDs[which(child_IDs == partner1_name)]
                    partner2_cells <- cell_IDs[which(child_IDs == partner2_name)]
                    # Compare child + partner1 vs. partner2
                    child_partner1_name <- paste0(child_name, ".", partner1_name)
                    child_partner1_cells <- c(child_cells, partner1_cells)
                    # If there is only 1 cell in partner 2, automatically merge
                    if (length(partner2_cells) == 1) {
                      comparison1_output <- list("result" = "merge")
                      comparison1_proceed <- FALSE
                    } else {
                      # Check whether clusters have been previously compared
                      previous_comparison <- CHOIR:::.checkComparisonRecords(cluster1_name = child_partner1_name,
                                                                     cluster1_cells = child_partner1_cells,
                                                                     cluster2_name = partner2_name,
                                                                     cluster2_cells = partner2_cells,
                                                                     comparison_records = comparison_records,
                                                                     type = "bridge")
                      # If they have, fill in result matrix
                      if (previous_comparison[["previously_compared"]] == TRUE) {
                        comparison1_output <- list("result" = previous_comparison[["result"]])
                        comparison1_proceed <- FALSE
                      } else {
                        comparison1_proceed <- TRUE
                      }
                    }
                    # Compare child + partner1 vs. partner2
                    child_partner2_name <- paste0(child_name, ".", partner2_name)
                    child_partner2_cells <- c(child_cells, partner2_cells)
                    # If there is only 1 cell in partner 1, automatically merge
                    if (length(partner1_cells) == 1) {
                      comparison2_output <- list("result" = "merge")
                      comparison2_proceed <- FALSE
                    } else {
                      # Check whether clusters have been previously compared
                      previous_comparison <- CHOIR:::.checkComparisonRecords(cluster1_name = child_partner2_name,
                                                                     cluster1_cells = child_partner2_cells,
                                                                     cluster2_name = partner1_name,
                                                                     cluster2_cells = partner1_cells,
                                                                     comparison_records = comparison_records,
                                                                     type = "bridge")
                      # If they have, fill in result matrix
                      if (previous_comparison[["previously_compared"]] == TRUE) {
                        comparison2_output <- list("result" = previous_comparison[["result"]])
                        comparison2_proceed <- FALSE
                      } else {
                        comparison2_proceed <- TRUE
                      }
                    }
                    if (comparison1_proceed == TRUE | comparison2_proceed == TRUE) {
                      # Determine which input & nn matrices to use
                      roots <- unlist(stringr::str_extract_all(c(child_name, partner1_name, partner2_name), "P\\d*"))
                      if (dplyr::n_distinct(roots) == 1) {
                        if (n_input_matrices > 1) {
                          use_input_matrix <- roots[1]
                        } else {
                          use_input_matrix <- "P0"
                        }
                        use_nn_matrix <- roots[1]
                      } else {
                        use_input_matrix <- "P0"
                        use_nn_matrix <- "P0"
                      }
                      if (comparison1_proceed == TRUE) {
                        comparison_start_time <- Sys.time()
                        child_partner1_comparison_names <- c(paste0(child_name, " vs. ", partner1_name),
                                                             paste0(partner1_name, " vs. ", child_name))
                        # Get distance between clusters
                        distance_check <- CHOIR:::.checkDistance(object = object,
                                                         key = key,
                                                         cluster1_name = child_partner1_name,
                                                         cluster1_cells = child_partner1_cells,
                                                         cluster2_name = partner2_name,
                                                         cluster2_cells = partner2_cells,
                                                         distance_awareness = distance_awareness,
                                                         distance_approx = distance_approx,
                                                         use_input_matrix = use_input_matrix,
                                                         dist_matrix = dist_matrix,
                                                         reduction = reduction,
                                                         distance_records = NULL)
                        # Run comparison 1
                        comparison1_output <- CHOIR:::.runPermutationTest(cluster1_name = child_partner1_name,
                                                                  cluster1_cells = child_partner1_cells,
                                                                  cluster1_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                               batches[child_partner1_cells],
                                                                                               NULL),
                                                                  cluster2_name = partner2_name,
                                                                  cluster2_cells = partner2_cells,
                                                                  cluster2_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                               batches[partner2_cells],
                                                                                               NULL),
                                                                  alpha = ifelse(p_adjust != "none", adjusted_alpha, alpha),
                                                                  n_iterations = n_iterations,
                                                                  n_trees = n_trees,
                                                                  use_variance = use_variance,
                                                                  min_accuracy = min_accuracy,
                                                                  min_connections = min_connections,
                                                                  max_repeat_errors = max_repeat_errors,
                                                                  collect_all_metrics = collect_all_metrics,
                                                                  sample_max = sample_max,
                                                                  downsampling_rate = downsampling_rate,
                                                                  input_matrix = input_matrices[[use_input_matrix]],
                                                                  nn_matrix = nn_matrices[[use_nn_matrix]],
                                                                  comparison_records = comparison_records,
                                                                  feature_importance_records = feature_importance_records,
                                                                  P0_distance = distance_check[["P0_distance"]],
                                                                  P_i_distance = distance_check[["P_i_distance"]],
                                                                  comparison_start_time = comparison_start_time,
                                                                  n_cores = n_cores,
                                                                  random_seed = random_seed)
                        # If split, add distances to distance_records
                        if (comparison1_output[["result"]] == "split" & methods::is(distance_awareness, "numeric")) {
                          distance_records <- CHOIR:::.addDistance(cluster1_name = child_partner1_name,
                                                           cluster2_name = partner2_name,
                                                           P0_distance = distance_check[["P0_distance"]],
                                                           P_i_distance = distance_check[["P_i_distance"]],
                                                           max_p = comparison1_output[["max_p"]],
                                                           distance_records = distance_records)
                        }
                        # Update records
                        comparison_records <- comparison1_output[["comparison_records"]]
                        feature_importance_records <- comparison1_output[["feature_importance_records"]]
                        comparison1 <- comparison_records %>% dplyr::filter(comparison == paste0(child_partner1_name, " vs. ", partner2_name))
                      }
                      if (comparison2_proceed == TRUE) {
                        comparison_start_time <- Sys.time()
                        child_partner2_comparison_names <- c(paste0(child_name, " vs. ", partner2_name),
                                                             paste0(partner2_name, " vs. ", child_name))
                        # Get distance between clusters
                        distance_check <- CHOIR:::.checkDistance(object = object,
                                                         key = key,
                                                         cluster1_name = child_partner2_name,
                                                         cluster1_cells = child_partner2_cells,
                                                         cluster2_name = partner1_name,
                                                         cluster2_cells = partner1_cells,
                                                         distance_awareness = distance_awareness,
                                                         distance_approx = distance_approx,
                                                         use_input_matrix = use_input_matrix,
                                                         dist_matrix = dist_matrix,
                                                         reduction = reduction,
                                                         distance_records = NULL)
                        # Run comparison 2
                        comparison2_output <- CHOIR:::.runPermutationTest(cluster1_name = child_partner2_name,
                                                                  cluster1_cells = child_partner2_cells,
                                                                  cluster1_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                               batches[child_partner2_cells],
                                                                                               NULL),
                                                                  cluster2_name = partner1_name,
                                                                  cluster2_cells = partner1_cells,
                                                                  cluster2_cell_batches = `if`(batch_correction_method == "Harmony",
                                                                                               batches[partner1_cells],
                                                                                               NULL),
                                                                  alpha = ifelse(p_adjust != "none", adjusted_alpha, alpha),
                                                                  n_iterations = n_iterations,
                                                                  n_trees = n_trees,
                                                                  use_variance = use_variance,
                                                                  min_accuracy = min_accuracy,
                                                                  min_connections = min_connections,
                                                                  max_repeat_errors = max_repeat_errors,
                                                                  sample_max = sample_max,
                                                                  downsampling_rate = downsampling_rate,
                                                                  collect_all_metrics = collect_all_metrics,
                                                                  input_matrix = input_matrices[[use_input_matrix]],
                                                                  nn_matrix = nn_matrices[[use_nn_matrix]],
                                                                  comparison_records = comparison_records,
                                                                  feature_importance_records = feature_importance_records,
                                                                  P0_distance = distance_check[["P0_distance"]],
                                                                  P_i_distance = distance_check[["P_i_distance"]],
                                                                  comparison_start_time = comparison_start_time,
                                                                  n_cores = n_cores,
                                                                  random_seed = random_seed)
                        # If split, add distances to distance_records
                        if (comparison2_output[["result"]] == "split" & methods::is(distance_awareness, "numeric")) {
                          distance_records <- CHOIR:::.addDistance(cluster1_name = child_partner2_name,
                                                           cluster2_name = partner1_name,
                                                           P0_distance = distance_check[["P0_distance"]],
                                                           P_i_distance = distance_check[["P_i_distance"]],
                                                           max_p = comparison2_output[["max_p"]],
                                                           distance_records = distance_records)
                        }
                        # Update records
                        comparison_records <- comparison2_output[["comparison_records"]]
                        feature_importance_records <- comparison2_output[["feature_importance_records"]]
                        comparison2 <- comparison_records %>% dplyr::filter(comparison == paste0(child_partner2_name, " vs. ", partner1_name))
                      }
                    }
                    # Update results matrix
                    if (comparison1_output[["result"]] == "merge" & comparison2_output[["result"]] == "split") {
                      result_matrix[child_name, partner1_name] <- "split"
                      result_matrix[partner1_name, child_name] <- "split"
                    } else if (comparison1_output[["result"]] == "split" & comparison2_output[["result"]] == "merge") {
                      result_matrix[child_name, partner2_name] <- "split"
                      result_matrix[partner2_name, child_name] <- "split"
                    } else if (comparison1_output[["result"]] == "split" & comparison2_output[["result"]] == "split") {
                      # If they both result in a "split", merge child cluster with the partner w/ the lowest distance
                      if (methods::is(distance_awareness, "numeric")) {
                        if (use_input_matrix == "P0") {
                          child_partner1_distance <- dplyr::filter(comparison_records,
                                                                   comparison %in% child_partner1_comparison_names)$root_distance
                          child_partner2_distance <- dplyr::filter(comparison_records,
                                                                   comparison %in% child_partner2_comparison_names)$root_distance
                        } else {
                          child_partner1_distance <- dplyr::filter(comparison_records,
                                                                   comparison %in% child_partner1_comparison_names)$subtree_distance
                          child_partner2_distance <- dplyr::filter(comparison_records,
                                                                   comparison %in% child_partner2_comparison_names)$subtree_distance
                        }
                        # If distance has not been calculated
                        if (length(child_partner1_distance) == 0) {
                          distance_check <- CHOIR:::.checkDistance(object = object,
                                                           key = key,
                                                           cluster1_name = child_name,
                                                           cluster1_cells = child_cells,
                                                           cluster2_name = partner1_name,
                                                           cluster2_cells = partner1_cells,
                                                           distance_awareness = distance_awareness,
                                                           distance_approx = distance_approx,
                                                           use_input_matrix = use_input_matrix,
                                                           dist_matrix = dist_matrix,
                                                           reduction = reduction,
                                                           distance_records = NULL)
                          if (use_input_matrix == "P0") {
                            child_partner1_distance <- distance_check[["P0_distance"]]
                          } else {
                            child_partner1_distance <- distance_check[["P_i_distance"]]
                          }
                        }
                        if (length(child_partner2_distance) == 0) {
                          distance_check <- CHOIR:::.checkDistance(object = object,
                                                           key = key,
                                                           cluster1_name = child_name,
                                                           cluster1_cells = child_cells,
                                                           cluster2_name = partner2_name,
                                                           cluster2_cells = partner2_cells,
                                                           distance_awareness = distance_awareness,
                                                           distance_approx = distance_approx,
                                                           use_input_matrix = use_input_matrix,
                                                           dist_matrix = dist_matrix,
                                                           reduction = reduction,
                                                           distance_records = NULL)
                          if (use_input_matrix == "P0") {
                            child_partner2_distance <- distance_check[["P0_distance"]]
                          } else {
                            child_partner2_distance <- distance_check[["P_i_distance"]]
                          }
                        }
                        # Compare
                        if (child_partner1_distance > child_partner2_distance) {
                          result_matrix[child_name, partner1_name] <- "split"
                          result_matrix[partner1_name, child_name] <- "split"
                        } else {
                          result_matrix[child_name, partner2_name] <- "split"
                          result_matrix[partner2_name, child_name] <- "split"
                        }
                      } else {
                        # Alternately, use accuracy scores
                        child_partner1_mean_accuracy <- dplyr::filter(comparison_records,
                                                                 comparison %in% child_partner1_comparison_names)$mean_accuracy
                        child_partner2_mean_accuracy <- dplyr::filter(comparison_records,
                                                                 comparison %in% child_partner2_comparison_names)$mean_accuracy
                        if (child_partner1_mean_accuracy > child_partner2_mean_accuracy) {
                          result_matrix[child_name, partner1_name] <- "split"
                          result_matrix[partner1_name, child_name] <- "split"
                        } else {
                          result_matrix[child_name, partner2_name] <- "split"
                          result_matrix[partner2_name, child_name] <- "split"
                        }
                      }
                    } # Else if both result in "merge", do nothing, allow bridge
                  }
                }
              }
            }
          }
          # Loop through child clusters again to find merge groups
          merge_group_list <- vector(mode = "list", length = n_child_clusters)
          # Change this if all children merge
          all_merge <- FALSE
          # For each child
          for (child in 1:n_child_clusters) {
            child_name <- unique_child_IDs[child]
            # Stop if it becomes established that all children will merge
            if (all_merge == FALSE) {
              # count cells
              n_merge <- sum(result_matrix[child_name,] == "merge", na.rm = TRUE)
              n_splits <- sum(result_matrix[child_name,] == "split", na.rm = TRUE)
              # if all merge
              if (n_merge == (n_child_clusters-1)) {
                all_merge <- TRUE
              } else if (n_splits == (n_child_clusters-1)) { # If all splits
                merge_group_list[[child]] <- child_name
              } else {
                merge_names <- rownames(result_matrix)[result_matrix[child_name,] == "merge"]
                merge_names <- merge_names[!is.na(merge_names)]
                merge_names <- c(child_name, merge_names)
                # Check if this group overlaps with any that are already identified
                if (child > 1) {
                  for (i in 1:(child-1)) {
                    overlap <- length(intersect(merge_names, merge_group_list[[i]]))
                    if (overlap > 0) {
                      # include all members & delete old group
                      merge_names <- unique(c(merge_group_list[[i]], merge_names))
                      merge_group_list[i] <- list(NULL)
                    }
                  }
                }
                merge_group_list[[child]] <- merge_names
              }
            }
          }
          # Convert merge groups to new cluster names
          new_labels_list <- CHOIR:::.getNewLabels(merge_groups = merge_group_list,
                                           parent_labels = evolving_parent_IDs)
          evolving_parent_IDs <- new_labels_list[["evolving_parent_IDs"]]
          merge_group_labels <- new_labels_list[["merge_group_labels"]]

          # Update child_IDs
          if (all_merge == TRUE) {
            child_IDs[parent_inds] <- parent_IDs[parent_inds]
          } else {
            # For each cell
            for (cell in 1:length(child_IDs)) {
              # If in the current parent branch
              if (cell %in% parent_inds) {
                # Check if cell is in any of the merge groups
                for (child in 1:n_child_clusters) {
                  if (child_IDs[cell] %in% merge_group_list[[child]]) {
                    child_IDs[cell] <- merge_group_labels[[child]]
                  }
                }
              }
            }
          }
        }
      } else {
        # No need to update child IDs if there is only one child cluster
        # (Cluster will retain same ID unless it merges, to retain ability to use records)
        # Progress
        if (lvl >= 0) {
          tick_amount <- 0.9*(1/length(unique_parent_IDs))*(0.9*level_weights[paste0("L", lvl)])
          pb$tick(tick_amount)
          if (verbose & ((((percent_done + tick_amount) %/% 10) - (percent_done %/% 10) > 0) |
                         (difftime(Sys.time(), hour_start_time, units = "hours") >= 1))) {
            hour_start_time <- Sys.time()
            pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"),
                              " : ", round((percent_done + tick_amount)), "% (", n_levels - lvl, "/", n_levels ," levels) in ",
                              round(difftime(Sys.time(), start_time, units = "min"), 2),
                              " min. ", dplyr::n_distinct(child_IDs), " clusters remaining."))
          }
          percent_done <- percent_done + tick_amount
        }
      }
      # Progress
      if (lvl >= 0) {
        tick_amount <- 0.1*(1/length(unique_parent_IDs))*(0.9*level_weights[paste0("L", lvl)])
        pb$tick(tick_amount)
        if (verbose & ((((percent_done + tick_amount) %/% 10) - (percent_done %/% 10) > 0) |
                       (difftime(Sys.time(), hour_start_time, units = "hours") >= 1))) {
          hour_start_time <- Sys.time()
          pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"),
                            " : ", round((percent_done + tick_amount)), "% (", n_levels - lvl, "/", n_levels ," levels) in ",
                            round(difftime(Sys.time(), start_time, units = "min"), 2),
                            " min. ", dplyr::n_distinct(child_IDs), " clusters remaining."))
        }
        percent_done <- percent_done + tick_amount
      }
    }
    # Check multiple comparison adjustment
    # Starting after at least 10 total comparisons & at least 3 "split" calls
    # Or if we're at the top of the tree with at least 1 "split" call
    if (p_adjust != "none" &
        ((lvl <= 1 & sum(comparison_records$decision %in% c("split", "split: repeat error")) >= 1) |
         (length(comparison_records$percentile_accuracy[!is.na(comparison_records$percentile_accuracy)]) >=
          ifelse(p_adjust == "fdr", 100, 50) &
          sum(comparison_records$decision %in% c("split", "split: repeat error")) >=
          ifelse(p_adjust == "fdr", 5, 1)))) {

      # Set a new alpha threshold
      p_accuracy <- comparison_records$percentile_accuracy[!is.na(comparison_records$percentile_accuracy)]
      p_variance <- comparison_records$percentile_variance[!is.na(comparison_records$percentile_variance)]
      if (p_adjust == "fdr") {
        # Based on FDR
        p_accuracy_adj <- p_accuracy[stats::p.adjust(p_accuracy, method = "fdr") < alpha]
        if (length(p_accuracy_adj) > 0) {
          p_accuracy_max <- (max(which(sort(p_accuracy) == max(p_accuracy_adj)))/length(p_accuracy))*alpha
        } else {
          p_accuracy_max <- adjusted_alpha
        }
        p_variance_adj <- p_variance[stats::p.adjust(p_variance, method = "fdr") < alpha]
        if (length(p_variance_adj) > 0) {
          p_variance_max <- (max(which(sort(p_variance) == max(p_variance_adj)))/length(p_variance))*alpha
        } else {
          p_variance_max <- adjusted_alpha
        }
        adjusted_alpha <- min(adjusted_alpha, max(p_variance_max, p_variance_max, na.rm = TRUE), na.rm = TRUE)
      } else if (p_adjust == "bonferroni") {
        adjusted_alpha <- alpha/length(p_accuracy)
      }
      # Remove distance records in which the comparison p-value was above the new adjusted alpha value
      distance_records <- dplyr::filter(distance_records, max_pval < adjusted_alpha)

      # Check for cases where p-values are now below alpha threshold
      # Where both clusters are still among the current cluster IDs
      if (use_variance == TRUE) {
        correction_check <- comparison_records %>%
          dplyr::mutate(correct = ifelse(decision == "split" &
                                           (percentile_accuracy >= adjusted_alpha |
                                              percentile_variance >= adjusted_alpha), TRUE,
                                         ifelse(decision == "split: repeat error" &
                                                  (percentile_modified_accuracy >= adjusted_alpha |
                                                     percentile_modified_variance >= adjusted_alpha), TRUE, FALSE))) %>%
          dplyr::filter(correct == TRUE) %>%
          tidyr::separate_wider_delim(comparison, delim = " vs. ", names = c("cluster1", "cluster2")) %>%
          dplyr::filter(cluster1 %in% unique(child_IDs),
                        cluster2 %in% unique(child_IDs))
      } else {
        correction_check <- comparison_records %>%
          dplyr::mutate(correct = ifelse(decision == "split" &
                                           percentile_accuracy >= adjusted_alpha, TRUE,
                                         ifelse(decision == "split: repeat error" &
                                                  percentile_modified_accuracy >= adjusted_alpha, TRUE, FALSE))) %>%
          dplyr::filter(correct == TRUE) %>%
          tidyr::separate_wider_delim(comparison, delim = " vs. ", names = c("cluster1", "cluster2")) %>%
          dplyr::filter(cluster1 %in% unique(child_IDs),
                        cluster2 %in% unique(child_IDs))
      }


      if (nrow(correction_check) > 0) {
        # Check whether comparisons should remain unmerged, from bottom up
        for (comp in 1:nrow(correction_check)) {
          # Check whether these clusters appear elsewhere in the corrections
          if (sum(c(correction_check$cluster1, correction_check$cluster2) == correction_check$cluster1[comp]) == 1 &
              sum(c(correction_check$cluster1, correction_check$cluster2) == correction_check$cluster2[comp]) == 1) {
            # If they do not, merge these two clusters
            merge_pair <- correction_check[comp,]
          } else {
            # If these clusters do appear elsewhere in the corrections that are among the current cluster IDs,
            # merge the cluster pair with the closest distance
            if (methods::is(distance_awareness, "numeric")) {
              subtree_distances <- dplyr::filter(correction_check,
                                                 cluster1 %in% c(correction_check$cluster1[comp],
                                                                 correction_check$cluster2[comp]) |
                                                   cluster2 %in% c(correction_check$cluster1[comp],
                                                                   correction_check$cluster2[comp]))$subtree_distance
              if (length(subtree_distances) > 0 & any(!is.na(subtree_distances))) {
                min_subtree_distance <- min(subtree_distances, na.rm = TRUE)
              } else {
                min_subtree_distance <- NA
              }
              if (!is.na(min_subtree_distance)) {
                # Use subtree distance if available
                merge_pair <- correction_check %>%
                  dplyr::filter(cluster1 %in% c(correction_check$cluster1[comp],
                                                correction_check$cluster2[comp]) |
                                  cluster2 %in% c(correction_check$cluster1[comp],
                                                  correction_check$cluster2[comp]),
                                subtree_distance == min_subtree_distance)
              } else {
                # Otherwise use root distance
                root_distances <- dplyr::filter(correction_check,
                                                cluster1 %in% c(correction_check$cluster1[comp],
                                                                correction_check$cluster2[comp]) |
                                                  cluster2 %in% c(correction_check$cluster1[comp],
                                                                  correction_check$cluster2[comp]))$root_distance
                min_root_distance <- min(root_distances, na.rm = TRUE)
                merge_pair <- correction_check %>%
                  dplyr::filter(cluster1 %in% c(correction_check$cluster1[comp],
                                                correction_check$cluster2[comp]) |
                                  cluster2 %in% c(correction_check$cluster1[comp],
                                                  correction_check$cluster2[comp]),
                                root_distance == min_root_distance)
              }
            } else {
              # Alternately, use accuracy scores
              min_accuracy <- min(dplyr::filter(correction_check,
                                                cluster1 %in% c(correction_check$cluster1[comp],
                                                                correction_check$cluster2[comp]) |
                                                  cluster2 %in% c(correction_check$cluster1[comp],
                                                                  correction_check$cluster2[comp]))$mean_accuracy,
                                  na.rm = TRUE)
              merge_pair <- correction_check %>%
                dplyr::filter(cluster1 %in% c(correction_check$cluster1[comp],
                                              correction_check$cluster2[comp]) |
                                cluster2 %in% c(correction_check$cluster1[comp],
                                                correction_check$cluster2[comp]),
                              mean_accuracy == min_accuracy)
            }
          }
          new_labels_list <- CHOIR:::.getNewLabels(merge_groups = list(c(merge_pair$cluster1[1],
                                                                 merge_pair$cluster2[1])),
                                           parent_labels = evolving_parent_IDs)
          evolving_parent_IDs <- new_labels_list[["evolving_parent_IDs"]]
          merged_label <- new_labels_list[["merge_group_labels"]][[1]]

          child_IDs[child_IDs %in% c(merge_pair$cluster1[1], merge_pair$cluster2[1])] <- merged_label
          # Update records
          comparison_records$decision[which(comparison_records$comparison ==
                                              paste0(merge_pair$cluster1[1], " vs. ",
                                                     merge_pair$cluster2[1]))] <- "merge: adjustment"
          n_current_clusters <- n_current_clusters - 1
        }
      }
    }
    # Update parent IDs
    if (lvl > 1) {
      # Move up clustering tree
      parent_IDs <- cluster_tree[, lvl-1]
    } else {
      # Continue until all clusters have been compared
      parent_IDs <- rep(paste0("P0_L", paste(rep(0, (((-1)*lvl) + 1)), collapse = ""), "_1"), length(cell_IDs))
    }
    # Record stepwise changes
    stepwise_child_IDs_df <- data.frame(stepwise_cluster_IDs = child_IDs)
    if (lvl >= 0) {
      colnames(stepwise_child_IDs_df) <- paste0("stepwise_cluster_ID_", alpha, "_L", lvl)
    } else {
      colnames(stepwise_child_IDs_df) <- paste0("stepwise_cluster_ID_", alpha, "_L", paste(rep(0, abs(lvl) + 1), collapse = ""))
    }
    stepwise_cluster_IDs <- cbind(stepwise_cluster_IDs, stepwise_child_IDs_df)
    # Progress
    if (lvl >= 0) {
      tick_amount <- 0.1*level_weights[paste0("L", lvl)]
      pb$tick(tick_amount)
      if (verbose & ((((percent_done + tick_amount) %/% 10) - (percent_done %/% 10) > 0) |
                     (difftime(Sys.time(), hour_start_time, units = "hours") >= 1))) {
        hour_start_time <- Sys.time()
        pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"),
                          " : ", round((percent_done + tick_amount)), "% (", n_levels - lvl, "/", n_levels ," levels) in ",
                          round(difftime(Sys.time(), start_time, units = "min"), 2),
                          " min. ", dplyr::n_distinct(child_IDs), " clusters remaining."))
      }
      percent_done <- percent_done + tick_amount
    }
    # Check for completion if beyond root of clustering tree
    if (lvl < 1) {
      if (underclustering_buffer == FALSE &
          (dplyr::n_distinct(stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)]) ==
           dplyr::n_distinct(paste(stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)], stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)-1])))) {
        # Check if any clusters from bottom of tree are still present
        check_df <- data.frame(original = cluster_tree[, n_levels],
                            current = stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)])
        check <- check_df %>%
          dplyr::group_by(current, original) %>%
          dplyr::filter(!(current %in% checked_for_underclustering)) %>%
          dplyr::filter(!(original %in% checked_for_underclustering)) %>%
          dplyr::summarise(cells = dplyr::n()) %>%
          dplyr::group_by(current) %>%
          dplyr::summarise(n = dplyr::n()) %>%
          dplyr::filter(n == 1)

        clusters_to_check <- unique(child_IDs)[unique(child_IDs) %in% results_of_underclustering_check]
        clusters_to_check <- clusters_to_check[!(clusters_to_check %in% checked_for_underclustering)]
        clusters_to_check <- c(clusters_to_check, unique(check$current))
        checked_for_underclustering <- unique(c(checked_for_underclustering,
                                         clusters_to_check,
                                         unique(dplyr::filter(check_df, current %in% check$current)$original)))

    
        complete <- TRUE
        pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"), " : Completed: all clusters compared."))
        pb$tick(5)
      } else if (dplyr::n_distinct(stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)]) == 1) {
        complete <- TRUE
        pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"), " : Completed: only one cluster remaining."))
        pb$tick(5)
      } else {
        pb$message(paste0(format(Sys.time(), "%Y-%m-%d %X"), " : Additional comparisons necessary. ",
                          dplyr::n_distinct(stepwise_cluster_IDs[, ncol(stepwise_cluster_IDs)]), " clusters remaining."))
        underclustering_buffer <- FALSE
      }
    }
    # Increment level
    lvl <- lvl - 1
  }

  # Finalize cluster IDs
  n_final_clusters <- dplyr::n_distinct(child_IDs)
  final_clusters <- data.frame(CellID = cell_IDs,
                               Record_cluster_label = child_IDs)
  # Convert to numeric cluster labels based on cluster size
  cluster_key <- final_clusters %>%
    dplyr::group_by(Record_cluster_label) %>%
    dplyr::summarise(n = dplyr::n()) %>%
    dplyr::arrange(-n)
  cluster_key$CHOIR_ID <- 1:n_final_clusters
  final_clusters <- merge(final_clusters, cluster_key, by = "Record_cluster_label", all.x = TRUE) %>%
    dplyr::select(CellID, CHOIR_ID, Record_cluster_label)
  colnames(final_clusters)[2] <- paste0("CHOIR_clusters_", alpha)

  # Rearrange back to original order
  rownames(final_clusters) <- final_clusters$CellID
  final_clusters <- final_clusters[cell_IDs, ]

  # Pull out accuracy scores for comparisons between final clusters
  final_cluster_mean_accuracies <- matrix(NA, nrow = n_final_clusters, ncol = n_final_clusters)
  final_cluster_distances <- matrix(NA, nrow = n_final_clusters, ncol = n_final_clusters)
  for (i in 1:(n_final_clusters-1)) {
    cluster_i_name <- cluster_key[cluster_key$CHOIR_ID == i, "Record_cluster_label"]
    for (j in (i+1):n_final_clusters) {
      cluster_j_name <- cluster_key[cluster_key$CHOIR_ID == j, "Record_cluster_label"]

      comparison_ij <- dplyr::filter(comparison_records,
                                  comparison == paste0(cluster_i_name, " vs. ", cluster_j_name) |
                                    comparison == paste0(cluster_j_name, " vs. ", cluster_i_name))
      if (nrow(comparison_ij) == 1) {
        final_cluster_mean_accuracies[i, j] <- comparison_ij$mean_accuracy
        final_cluster_mean_accuracies[j, i] <- comparison_ij$mean_accuracy
        final_cluster_distances[i, j] <- comparison_ij$root_distance
        final_cluster_distances[j, i] <- comparison_ij$root_distance
      }
    }
  }

  # Add to object
  object <- CHOIR:::.storeData(object, key, "final_clusters",
                       data.frame(CHOIR_IDs = final_clusters[, paste0("CHOIR_clusters_", alpha)]),
                       paste0("CHOIR_clusters_", alpha))
  object <- CHOIR:::.storeData(object, key, "clusters", final_clusters, paste0("CHOIR_clusters_", alpha))
  object <- CHOIR:::.storeData(object, key, "clusters", stepwise_cluster_IDs, paste0("stepwise_clusters_", alpha))
  object <- CHOIR:::.storeData(object, key, "records", final_cluster_mean_accuracies, paste0("CHOIR_clusters_", alpha, "_accuracies"))
  object <- CHOIR:::.storeData(object, key, "records", final_cluster_distances, paste0("CHOIR_clusters_", alpha, "_distances"))

  # -------------------------------------------------------------------------
  # Report results & warnings
  # -------------------------------------------------------------------------

  if (verbose) {
    message("")
    if (max_repeat_errors > 0 & sum(comparison_records$decision == "split: repeat error") > 0) {
      message(" - Repeatedly misassigned cells affected the result of ", sum(comparison_records$decision == "split: repeat error"),
              " comparisons. Set 'max_repeat_errors' parameter to 0 if you would like to disable this setting.")
    }
    if (methods::is(distance_awareness, "numeric") & sum(comparison_records$decision == "split: distance") > 0) {
      message(" - In ", sum(comparison_records$decision == "split: distance"),
              " comparisons, clusters were split due to cluster distance threshold.")
    }
    if (min_connections > 0 & sum(comparison_records$decision == "split: min connections") > 0) {
      message(" - In ", sum(comparison_records$decision == "split: min connections"),
              " comparisons, clusters were split due to the minimum number of nearest neighbor connections.")
    }
    if (sum(comparison_records$decision == "merge: min accuracy") > 0) {
      message(" - In ", sum(comparison_records$decision == "merge: min accuracy"),
              " comparisons, clusters were merged due to the minimum accuracy threshold.")
    }
    if (p_adjust != "none" & sum(comparison_records$decision == "merge: adjustment") > 0) {
      message(" - In ", sum(comparison_records$decision == "merge: adjustment"),
              " comparisons, clusters were merged after correction for multiple comparisons.")
      message(" - Final adjusted significance threshold = ", ifelse(adjusted_alpha > 0.0001, round(adjusted_alpha, 5), signif(adjusted_alpha, digits=5)), ".")
    }
    if (sum(comparison_records$decision == "merge: batch-dependent") > 0) {
      message(" - In ", sum(comparison_records$decision == "merge: batch-dependent"),
              " comparisons, clusters were merged due to batch-dependence.")
    }
    message("\n", format(Sys.time(), "%Y-%m-%d %X"), " : Identified ", n_final_clusters, " clusters.")
  }

  # Record parameters used and add to original object
  parameter_list <- list("subtree_names" = names(input_matrices),
                         "alpha" = alpha,
                         "p_adjust_method" = p_adjust,
                         "adjusted_alpha" = adjusted_alpha,
                         "feature_set" = feature_set,
                         "exclude_features" = exclude_features,
                         "n_iterations" = n_iterations,
                         "n_trees" = n_trees,
                         "use_variance" = use_variance,
                         "min_accuracy" = min_accuracy,
                         "min_connections" = min_connections,
                         "max_repeat_errors" = max_repeat_errors,
                         "distance_approx" = distance_approx,
                         "distance_awareness" = distance_awareness,
                         "collect_all_metrics" = collect_all_metrics,
                         "sample_max" = sample_max,
                         "downsampling_rate" = downsampling_rate,
                         "normalization_method" = normalization_method,
                         "batch_correction_method" = batch_correction_method,
                         "batch_labels" = batch_labels,
                         "cluster_tree_provided" = cluster_tree_provided,
                         "input_matrix_provided" = input_matrix_provided,
                         "nn_matrix_provided" = nn_matrix_provided,
                         "dist_matrix_provided" = dist_matrix_provided,
                         "reduction_provided" = reduction_provided,
                         "random_seed" = random_seed)

  object <- CHOIR:::.storeData(object, key, "parameters", parameter_list, "pruneTree_parameters")

  # Add records to object
  object <- CHOIR:::.storeData(object, key, "records", comparison_records, "comparison_records")
  if (collect_all_metrics == TRUE) {
    object <- CHOIR:::.storeData(object, key, "records", feature_importance_records, "feature_importance_records")
  }

  # Return object with new additions
  return(object)
}

In [ ]:
%%R

# Some parameters must be manually set to avoid the function crashing
object_type = "SingleCellExperiment"
n_modalities = 1
cluster_params = list('algorithm'=4,'method'='igraph')
parameter_list <- list("cluster_params" = cluster_params,
                         "random_seed" = 66
)

# add metadata slot for storing results, and add manual parameters
sce@metadata[["CHOIR"]] <- list()
sce@metadata[["CHOIR"]][["parameters"]][['buildTree_parameters']] <- parameter_list

In [ ]:
%%time
%%R

# prune the cluster tree
sce <- OverpruneTree(sce, 
                 cluster_tree = hc_tree,
                 input_matrix = input_matrix, 
                 nn_matrix = nn_mat,
                 reduction = reduction,
                 n_cores = parallel::detectCores() - 2,
                 random_seed = 66,
                verbose = TRUE
                )

In [ ]:
%%R -o choir_cluster_labels

# extract final cluster labels
choir_cluster_labels <- colData(sce)$CHOIR_clusters_0.05

In [ ]:
# add labels to anndata object
human.obs['choir_clusters'] = choir_cluster_labels
human.obs['choir_clusters'] = human.obs['choir_clusters'].astype('category')

In [ ]:
# plot umap with final cluster labels
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    sc.pl.umap(human, color=['choir_clusters'], 
           legend_loc='on data',
           legend_fontsize = 'xx-small',
           legend_fontweight='medium',
           legend_fontoutline=1,
           title='',
           frameon = False)

In [ ]:
# plot umap with final cluster labels
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    sc.pl.umap(human, color=['choir_clusters'], 
           legend_loc='right margin',
           title='',
           frameon = False)

In [ ]:
# calculate cluster marker genes for manual annotation
sc.tl.rank_genes_groups(human, groupby = "choir_clusters", method='wilcoxon',
                        use_raw = False, layer = "log1p_norm", pts = True, 
                        key_added = "choir_clusters")

In [ ]:
# save cell type marker genes DE test results
temp_df = sc.get.rank_genes_groups_df(human, key = "choir_clusters", group = None)
temp_df.to_csv("../output/human/human_cluster_wilcoxon_test_results.csv")
temp_df.to_pickle("../output/human/human_cluster_wilcoxon_test_results.pkl")
temp_df.loc[(abs(temp_df['logfoldchanges']) > 1) & (temp_df['pvals_adj'] < 0.05)].to_csv("../output/human/human_significant_cluster_markers.csv")
temp_df.loc[(abs(temp_df['logfoldchanges']) > 1) & (temp_df['pvals_adj'] < 0.05)].to_pickle("../output/human/human_significant_cluster_markers.pkl")

In [ ]:
# make list of unique marker genes which define choir clusters
unique_marker_genes = np.unique(temp_df.loc[(abs(temp_df['logfoldchanges']) > 1) & (temp_df['pvals_adj'] < 0.05)].names)

In [ ]:
# save anndata with choir clustering information
human.write('../output/human/human_allcelltypes_choir_clustered_scanpy_object.h5ad')

In [ ]:
%%R
# save mrtree hierarchical clustering tree
saveRDS(hier_cluster_tree, "../output/human/human_allcelltypes_mrtree_hierarchical_clustertree.rds")

In [ ]:
%%R
# save CHOIR SCE object
saveRDS(sce, "../output/human/human_allcelltypes_choir_clustering_sce_output_object.rds")

In [ ]:
sc.tl.dendrogram(human, groupby = "choir_clusters", n_pcs = 90, use_rep = "X_pca_harmony", optimal_ordering = True)

In [ ]:
# plot umap with final cluster labels
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    sc.pl.umap(human, color=['choir_clusters'], 
           legend_loc='on data',
           legend_fontsize = 'xx-small',
           legend_fontweight='medium',
           legend_fontoutline=1,
           title='',
           frameon = False)

In [ ]:
cmap = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap = mpl.colors.ListedColormap(cmap[25:,:-1])

cmap2 = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap2 = mpl.colors.ListedColormap(cmap2[0:,:-1])

In [ ]:
# plot dotplot with cluster/cell type marker genes

cmap2 = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap2 = mpl.colors.ListedColormap(cmap2[0:,:-1])

with rc_context({"figure.figsize": (15, 12), "grid.alpha":0}): 
    sc.pl.dotplot(human, var_names = [ 
                                    'ASPA','MOG', # mature oligodendrocytes
                                    'PTPRC', # myeloid lineage cells
                                        'CX3CR1', # microglia
                                      'CSF1R', # macrophages
                                        'CLDN5','PECAM1', # endothelial cells
                                    'DCN','OGN', # VLMCs
                                    'KCNJ8', # pericytes

                                        'AQP4','ALDH1L1', # astrocytes

                                      'PDGFRA','CSPG4',  # OPCs
                                        'GPR17', # COPs
                                     'SYT1','RBFOX3' # mature neurons
                                     ], 
                  groupby = 'choir_clusters', standard_scale='var',
                  colorbar_title='Mean scaled\nexpression',
                  size_title='Fraction of cells\nwith expression (%)',
                  cmap=cmap2,
                  swap_axes = True,layer = 'log1p_norm', dendrogram = True)

In [ ]:
#annotate clusters with cell type labels
human.obs['annotation_level1'] = (
    human.obs["choir_clusters"]
    .map(lambda x: {1:"Astrocytes", 
                    2:"Astrocytes",
                    3:"Microglia",
                    4:"Neurons",
                    5:"Oligodendrocytes",
                    6:"OPCs",
                    7:"Oligodendrocytes",
                    8:"Astrocytes",
                    9:"Neurons",
                    10:"Neurons",
                    11:"Neurons",
                    12:"Neurons",
                    13:"Neurons",
                    14:"OPCs",
                    15:"Neurons",
                    16:"Astrocytes",
                    17:"Neurons",
                    18:"Neurons",
                    19:"OPCs",
                    20:"Neurons",
                    21:"Neurons",
                    22:"Astrocytes",
                    23:"Neurons",
                    24:"Neurons",
                    25:"Neurons",
                    26:"Neurons",
                    27:"Neurons",
                    28:"Neurons",
                    29:"Neurons",
                    30:"Neurons",
                    31:"Neurons",
                    32:"Neurons",
                    33:"Neurons",
                    34:"Neurons",
                    35:"Endothelial",
                    36:"Neurons",
                    37:"Neurons",
                    38:"Oligodendrocytes",
                    39:"Neurons",
                    40:"Astrocytes",
                    41:"Neurons",
                    42:"Neurons",
                    43:"Neurons",
                    44:"Neurons",
                    45:"Neurons",
                    46:"Neurons",
                    47:"OPCs",
                    48:"Neurons",
                    49:"OPCs",
                    50:"Neurons",
                    51:"Neurons",
                    52:"Neurons",
                    53:"Neurons",
                    54:"Neurons",
                    55:"Astrocytes",
                    56:"Astrocytes",
                    57:"COPs",
                    58:"Neurons",
                    59:"Pericytes",
                    60:"Astrocytes",
                    61:"VLMCs",
                    62:"Microglia"             
                   }.get(x, x))
    .astype("category")
)

In [ ]:
# re-order cell type labels
human.obs['annotation_level1'] = human.obs['annotation_level1'].cat.reorder_categories(['Microglia',
                                                                                        'Astrocytes',
                                                                                       "Oligodendrocytes",
                                                                                       "COPs",
                                                                                       "OPCs",
                                                                                       "Neurons",
                                                                                       "Endothelial",
                                                                                       "Pericytes",
                                                                                       "VLMCs"])

In [ ]:
# plot umap with final cell type labels
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    sc.pl.umap(human, color=['annotation_level1'], 
           legend_loc='right margin', size = 10,
               palette = {'Microglia':'#1b9e77',
                         'Astrocytes':'#d95f02',
                         'Oligodendrocytes':'#7570b3',
                         'COPs':'#e7298a',
                         'OPCs':'#66a61e',
                         'Neurons':'#e6ab02',
                         'Endothelial':'#a6761d',
                         'Pericytes':'#000000',
                         'VLMCs':'#666666'
                         },
           title='',
           frameon = False)

In [ ]:
%%R

library(speckle)

In [ ]:
# extract metadata for creating SCE object
cluster_labels = human.obs['annotation_level1']
sample_labels = human.obs['sampleid']
group_labels = human.obs['condition']

In [ ]:
%%R -i cluster_labels -i sample_labels -i group_labels
# create SCE object
sce <- SingleCellExperiment(list(counts=matrix(ncol = length(sample_labels), nrow = 1)),
                     colData=data.frame(clusters=cluster_labels,
                                        sample=sample_labels,
                                        group=group_labels))

In [ ]:
%%R -w 4 -h 4 -r 300 --units in

color_dict <- c('Microglia'='#1b9e77ff',
 'Astrocytes'='#d95f02ff',
 'Oligodendrocytes'='#7570b3ff',
 'COPs'='#e7298aff',
 'OPCs'='#66a61eff',
 'Neurons'='#e6ab02ff',
 'Endothelial'='#a6761dff',
 'Pericytes'='#000000ff',
 'VLMCs'='#666666ff')

# Plot cell type proportions
plotCellTypeProps(clusters = colData(sce)$clusters, sample = colData(sce)$sample) + theme(aspect.ratio = 0.25) +
    scale_x_discrete(limits = c('Control_A', 'Control_B', 'Control_C', 'CART_A', 'CART_B', 'CART_C', 'CART_D')) + 
  scale_fill_manual(values=color_dict) + guides(fill = "none") + 
    theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
%%R -w 10 -h 4 -r 300 --units in

# Plot cell type proportions
plotCellTypeProps(clusters = colData(sce)$clusters, sample = colData(sce)$sample) + theme(aspect.ratio = 0.25) +
    scale_x_discrete(limits = c('Control_A', 'Control_B', 'Control_C', 'CART_A', 'CART_B', 'CART_C', 'CART_D')) + 
  scale_fill_manual(values=color_dict) + guides() + 
    theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
cmap = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap = mpl.colors.ListedColormap(cmap[25:,:-1])

cmap2 = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap2 = mpl.colors.ListedColormap(cmap2[0:,:-1])

In [ ]:
# plot dotplot with cluster/cell type marker genes

cmap2 = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap2 = mpl.colors.ListedColormap(cmap2[0:,:-1])

with rc_context({"figure.figsize": (15, 12), "grid.alpha":0}): 
    dp = sc.pl.dotplot(human, var_names = [ 
                                    'ASPA','MOG', # mature oligodendrocytes
                                    'PTPRC', # myeloid lineage cells
                                        'CX3CR1', # microglia
                                      'CSF1R', # macrophages
                                        'CLDN5','PECAM1', # endothelial cells
                                    'DCN','OGN', # VLMCs
                                    'KCNJ8', # pericytes

                                        'AQP4','ALDH1L1', # astrocytes

                                      'PDGFRA','CSPG4',  # OPCs
                                        'GPR17', # COPs
                                     'SYT1','RBFOX3' # mature neurons
                                     ], 
                  groupby = 'choir_clusters', standard_scale='var',
                  colorbar_title='Mean scaled\nexpression',
                  size_title='Fraction of cells\nwith expression (%)',
                  cmap=cmap2,
                  swap_axes = True,layer = 'log1p_norm', dendrogram = True,
                 show = False)


    ax = dp['mainplot_ax']
    ax2 = dp['size_legend_ax']

    # make gene names italic
    for label in ax.get_yticklabels():
        label.set_fontstyle('italic')

In [ ]:
# create anndata object of microglial cells
micros = human[human.obs['annotation_level1'].isin(['Microglia']), :].copy()
micros.X = micros.layers["raw_counts"].copy()
micros.write('../output/human/human_microglia_scanpy_object.h5ad')

In [ ]:
micros

In [ ]:
# create anndata object of oligodendroglial lineage cells
oligos = human[human.obs['annotation_level1'].isin(['OPCs', 'COPs', 'Oligodendrocytes']), :].copy()
oligos.X = oligos.layers["raw_counts"].copy()
oligos.write('../output/human/human_oligodendrocytes_scanpy_object.h5ad')

In [ ]:
oligos

In [ ]:
# create anndata object of astrocytes
astros = human[human.obs['annotation_level1'].isin(['Astrocytes']), :].copy()
astros.X = astros.layers["raw_counts"].copy()
astros.write('../output/human/human_astrocytes_scanpy_object.h5ad')

In [ ]:
astros

In [ ]:
# calculate cluster marker genes for manual annotation
sc.tl.rank_genes_groups(human, groupby = "annotation_level1", method='wilcoxon',
                        use_raw = False, layer = "log1p_norm", pts = True, 
                        key_added = "celltypes")

In [ ]:
# save cell type marker genes DE test results
temp_df = sc.get.rank_genes_groups_df(human, key = "celltypes", group = None)
temp_df.to_csv("../output/human/human_celltypes_wilcoxon_test_results.csv")
temp_df.to_pickle("../output/human/human_celltypes_wilcoxon_test_results.pkl")
temp_df.loc[(abs(temp_df['logfoldchanges']) > 1) & (temp_df['pvals_adj'] < 0.05)].to_csv("../output/human/human_significant_celltypes_markers.csv")
temp_df.loc[(abs(temp_df['logfoldchanges']) > 1) & (temp_df['pvals_adj'] < 0.05)].to_pickle("../output/human/human_significant_celltypes_markers.pkl")

In [ ]:
# save anndata with choir clustering information
human.write('../output/human/human_allcelltypes_choir_clustered_annotated_scanpy_object.h5ad')